# Python Logging and Debugging Techniques

This notebook covers comprehensive logging and debugging techniques in Python, including:
- Setting up the Python logging system
- Configuring loggers, handlers, and formatters
- Using logging in applications
- Leveraging debugging tools and techniques
- Performance profiling

Let's start by installing the required packages.

In [ ]:
# Install necessary packages
!pip install memory_profiler line_profiler rich loguru pytest pytest-cov

In [ ]:
# Import standard libraries
import logging
import sys
import io
import os
import time
import traceback
import random
import warnings
from datetime import datetime

## 1. Setting Up Python Logging

The `logging` module is part of Python's standard library and provides a flexible framework for emitting log messages from Python programs.

In [ ]:
# Basic logging setup
import logging

# The simplest way to use logging
logging.debug('This is a debug message')  # Won't be printed by default
logging.info('This is an info message')   # Won't be printed by default
logging.warning('This is a warning message')  # Will be printed
logging.error('This is an error message')  # Will be printed
logging.critical('This is a critical message')  # Will be printed

## 2. Logging Levels and Basic Configuration

Python logging has five standard levels indicating the severity of events. Each has a numeric value:

- DEBUG (10): Detailed information, typically only valuable when diagnosing problems.
- INFO (20): Confirmation that things are working as expected.
- WARNING (30): An indication that something unexpected happened, or may happen in the near future.
- ERROR (40): Due to a more serious problem, the software has not been able to perform some function.
- CRITICAL (50): A very serious error, indicating that the program itself may be unable to continue running.

In [ ]:
# Configure the logging level
logging.basicConfig(
    level=logging.DEBUG,  # Set the threshold for this logger to DEBUG
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Now all messages will be displayed
logging.debug('This debug message will now be displayed')
logging.info('This info message will now be displayed')
logging.warning('This warning message will be displayed')
logging.error('This error message will be displayed')
logging.critical('This critical message will be displayed')

## 3. Configuring Log Handlers and Formatters

Handlers are responsible for dispatching log messages to specific destinations like console, files, email, etc. Formatters specify the layout of log records in the final output.

In [ ]:
# Reset the root logger
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Create a custom logger
logger = logging.getLogger('my_logger')
logger.setLevel(logging.DEBUG)

# Create handlers
c_handler = logging.StreamHandler()  # Console handler
c_handler.setLevel(logging.WARNING)  # Set level for this handler

# Create formatters and add them to handlers
c_format = logging.Formatter('%(name)s - %(levelname)s - %(message)s')
c_handler.setFormatter(c_format)

# Add handlers to the logger
logger.addHandler(c_handler)

# Test the logger
logger.debug('This is a debug message')  # Won't be printed
logger.warning('This is a warning message')  # Will be printed

## 4. Logging to Files

Logging to files is a common practice for keeping a persistent record of events.

In [ ]:
# Create a file handler
try:
    # Create a new logger to avoid adding handlers to existing ones
    file_logger = logging.getLogger('file_logger')
    file_logger.setLevel(logging.DEBUG)
    
    # Create file handler
    f_handler = logging.FileHandler('app.log')
    f_handler.setLevel(logging.DEBUG)
    
    # Create formatter
    f_format = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    f_handler.setFormatter(f_format)
    
    # Add handler to logger
    file_logger.addHandler(f_handler)
    
    # Log messages
    file_logger.debug('Debug message is logged to file')
    file_logger.info('Info message is logged to file')
    file_logger.warning('Warning message is logged to file')
    
    print("Messages have been logged to 'app.log'")
    
    # Display the content of the log file
    with open('app.log', 'r') as f:
        print("\nLog file content:")
        print(f.read())
        
except Exception as e:
    print(f"An error occurred: {e}")

## 5. Creating Custom Loggers

In larger applications, it's common to create multiple loggers for different components.

In [ ]:
# Create loggers for different components
db_logger = logging.getLogger('database')
api_logger = logging.getLogger('api')
ui_logger = logging.getLogger('user_interface')

# Set log levels
db_logger.setLevel(logging.DEBUG)
api_logger.setLevel(logging.INFO)
ui_logger.setLevel(logging.WARNING)

# Create a console handler
console = logging.StreamHandler()
console.setLevel(logging.DEBUG)

# Configure formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
console.setFormatter(formatter)

# Add the handler to all loggers
db_logger.addHandler(console)
api_logger.addHandler(console)
ui_logger.addHandler(console)

# Test the loggers
db_logger.debug('Database connection established')
api_logger.info('API request received')
ui_logger.warning('UI component loading slowly')

## 6. Logging in Applications and Modules

Best practices for using logging in larger applications and modules.

In [ ]:
# Create a module-level logger
# Typically this would be in a separate .py file

# Simulating a module here
def setup_logger():
    # Create logger based on the module name
    module_logger = logging.getLogger('my_module')
    module_logger.setLevel(logging.DEBUG)
    
    # Only add handler if none exist
    if not module_logger.handlers:
        # Create handler
        handler = logging.StreamHandler()
        handler.setLevel(logging.DEBUG)
        
        # Create formatter
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        handler.setFormatter(formatter)
        
        # Add handler to logger
        module_logger.addHandler(handler)
    
    return module_logger

# Get the module logger
module_logger = setup_logger()

# A function in the module
def process_data(data):
    module_logger.debug(f"Processing {len(data)} items")
    result = []
    for i, item in enumerate(data):
        try:
            processed = item * 2  # Simple processing
            result.append(processed)
            module_logger.debug(f"Processed item {i}: {item} -> {processed}")
        except Exception as e:
            module_logger.error(f"Error processing item {i}: {e}")
    
    module_logger.info(f"Processed {len(result)} items successfully")
    return result

# Use the module function
data = [1, 2, 3, 4, 5]
processed_data = process_data(data)
print(f"Result: {processed_data}")

## 7. Exception Handling with Logging

Logging exceptions provides valuable information for debugging.

In [ ]:
# Setup a logger for this section
exception_logger = logging.getLogger('exception_handling')
exception_logger.setLevel(logging.DEBUG)

# Create handler if none exist
if not exception_logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(levelname)s:%(name)s:%(message)s')
    handler.setFormatter(formatter)
    exception_logger.addHandler(handler)

# Function that might raise an exception
def divide(x, y):
    try:
        result = x / y
        exception_logger.debug(f"Division {x}/{y} = {result}")
        return result
    except ZeroDivisionError:
        # Log exception with details
        exception_logger.error("Division by zero attempted", exc_info=True)
        # Or alternatively use exception() which automatically adds exc_info=True
        # exception_logger.exception("Division by zero attempted")
        return None
    except Exception as e:
        exception_logger.critical(f"Unexpected error: {e}", exc_info=True)
        return None

# Test the function
print(divide(10, 2))  # Should work
print(divide(10, 0))  # Should log an error

## 8. Using the Python Debugger (pdb)

Python's built-in debugger (pdb) is a powerful tool for interactive debugging.

In [ ]:
import pdb

# Create a function with a bug to debug
def calculate_sum(numbers):
    total = 0
    for num in numbers:
        # Imagine this is more complex and has a bug
        total += num
    return total

# Function to demonstrate pdb usage
def demo_pdb():
    print("Starting the function")
    numbers = [1, 2, 3, 4, 5]
    
    # Set a breakpoint programmatically
    # pdb.set_trace()  # Uncomment to enable debugging
    
    # Calculate the sum
    result = calculate_sum(numbers)
    print(f"The result is: {result}")

# Run the function
demo_pdb()

Common pdb commands:
* `h` (help): Show available commands
* `l` (list): Show the current line and surrounding code
* `n` (next): Execute the current line and move to the next line
* `s` (step): Step into a function call
* `c` (continue): Continue execution until the next breakpoint
* `p expr` (print): Print the value of an expression
* `q` (quit): Exit the debugger and program

Starting Python 3.7, you can also use the built-in `breakpoint()` function instead of `pdb.set_trace()`.

In [ ]:
# Using the breakpoint() function (Python 3.7+)
def demo_breakpoint():
    x = 10
    y = 5
    # breakpoint()  # Uncomment to enable debugging
    z = x + y
    return z

result = demo_breakpoint()
print(f"Result: {result}")

## 9. Debugging with Breakpoints

Conditional breakpoints and other advanced breakpoint techniques.

In [ ]:
# Function with a loop to demonstrate conditional breakpoints
def process_large_list(items):
    results = []
    for i, item in enumerate(items):
        # In pdb, you could set a conditional breakpoint with:
        # (Pdb) break 8, item > 50
        # This would pause only when item > 50
        
        # Simulate processing
        processed = item * 2
        results.append(processed)
        
        # Manually implement a conditional breakpoint for demonstration
        if item > 50:  # Only break for large values
            # breakpoint()  # Uncomment to enable debugging
            pass
            
    return results

# Create a list of numbers
numbers = [10, 20, 30, 40, 50, 60, 70, 80]
results = process_large_list(numbers)
print(f"First few results: {results[:3]}...")

## 10. Assertions for Debugging

Assertions are boolean expressions that check if conditions are as expected.

In [ ]:
def calculate_average(numbers):
    # Verify input is a list
    assert isinstance(numbers, list), "Input must be a list"
    
    # Verify list is not empty
    assert len(numbers) > 0, "Input list cannot be empty"
    
    # Verify all elements are numbers
    assert all(isinstance(num, (int, float)) for num in numbers), "All elements must be numbers"
    
    # Calculate average
    total = sum(numbers)
    average = total / len(numbers)
    
    # Verify calculation result
    assert average <= max(numbers), "Average cannot be greater than the maximum value"
    
    return average

# Test with valid data
try:
    print(f"Average of [1, 2, 3, 4, 5]: {calculate_average([1, 2, 3, 4, 5])}")
except AssertionError as e:
    print(f"Assertion failed: {e}")
    
# Test with invalid data
try:
    print(f"Average of empty list: {calculate_average([])}")
except AssertionError as e:
    print(f"Assertion failed: {e}")

# Test with non-list input
try:
    print(f"Average of 'not a list': {calculate_average('not a list')}")
except AssertionError as e:
    print(f"Assertion failed: {e}")

## 11. Using Logging for Debugging

Logging is a powerful tool for debugging, especially for scenarios where interactive debugging is not practical.

In [ ]:
# Configure logging for debugging
debug_logger = logging.getLogger('debug_logger')
debug_logger.setLevel(logging.DEBUG)

# Create handlers if none exist
if not debug_logger.handlers:
    # Console handler for immediate feedback
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.DEBUG)
    console_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(console_formatter)
    debug_logger.addHandler(console_handler)
    
    # File handler for persistent logs
    file_handler = logging.FileHandler('debug.log')
    file_handler.setLevel(logging.DEBUG)
    file_formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(file_formatter)
    debug_logger.addHandler(file_handler)

# A function with debug logging
def complex_calculation(a, b, c):
    debug_logger.debug(f"Starting calculation with inputs: a={a}, b={b}, c={c}")
    
    # First step
    intermediate1 = a * b
    debug_logger.debug(f"Intermediate result 1: {intermediate1}")
    
    # Second step
    try:
        intermediate2 = intermediate1 / c
        debug_logger.debug(f"Intermediate result 2: {intermediate2}")
    except ZeroDivisionError:
        debug_logger.error("Division by zero detected")
        return None
    
    # Final step
    result = intermediate2 + a
    debug_logger.debug(f"Final result: {result}")
    
    return result

# Test the function
result1 = complex_calculation(5, 10, 2)
print(f"Result 1: {result1}")

result2 = complex_calculation(5, 10, 0)  # This will cause division by zero
print(f"Result 2: {result2}")

## 12. Performance Profiling

Profiling helps identify performance bottlenecks in your code.

In [ ]:
# Simple timing using time module
import time

def measure_time(func, *args, **kwargs):
    start_time = time.time()
    result = func(*args, **kwargs)
    end_time = time.time()
    print(f"Function {func.__name__} took {end_time - start_time:.6f} seconds to execute")
    return result

# A function to profile
def slow_function(n):
    total = 0
    for i in range(n):
        total += i ** 2
    return total

# Profile the function
measure_time(slow_function, 1000000)

In [ ]:
# Using Python's built-in timeit module
import timeit

# Define functions to compare
def method1(n):
    return [i ** 2 for i in range(n)]

def method2(n):
    result = []
    for i in range(n):
        result.append(i ** 2)
    return result

# Setup code
setup = "from __main__ import method1, method2"

# Compare the two methods
n = 1000
t1 = timeit.timeit(f"method1({n})", setup=setup, number=1000)
t2 = timeit.timeit(f"method2({n})", setup=setup, number=1000)

print(f"Method 1 took {t1:.6f} seconds for 1000 iterations")
print(f"Method 2 took {t2:.6f} seconds for 1000 iterations")
print(f"Method 1 is {t2/t1:.2f}x faster than Method 2" if t1 < t2 else f"Method 2 is {t1/t2:.2f}x faster than Method 1")

In [ ]:
# Using cProfile for more detailed profiling
import cProfile
import pstats
from pstats import SortKey

# A more complex function to profile
def complex_function():
    result = 0
    for i in range(1000):
        result += sum(j * j for j in range(i))
    return result

# Profile the function
cProfile.run('complex_function()', 'profile_stats')

# Display the profiling results
p = pstats.Stats('profile_stats')
p.strip_dirs().sort_stats(SortKey.TIME).print_stats(10)  # Show top 10 functions by time

In [ ]:
# Memory profiling (requires memory_profiler package)
try:
    from memory_profiler import profile
    
    @profile
    def memory_intensive_function():
        # Create a large list
        large_list = [i for i in range(1000000)]
        # Process the list
        result = sum(large_list)
        # Create another large object
        large_dict = {i: i*i for i in range(100000)}
        # Free the list
        del large_list
        # Return result
        return result
    
    # Run the function
    memory_intensive_function()
    
except ImportError:
    print("memory_profiler not installed. Install it with 'pip install memory_profiler'")

## 13. Advanced Debugging Techniques

Some advanced techniques for debugging complex issues.

In [ ]:
# Post-mortem debugging
import pdb

def function_with_error():
    a = 10
    b = 0
    return a / b  # This will raise a ZeroDivisionError

try:
    result = function_with_error()
except Exception as e:
    traceback_details = traceback.format_exc()
    print(f"Caught an exception: {e}")
    print("\nTraceback:")
    print(traceback_details)
    
    # To start post-mortem debugging, uncomment this line:
    # pdb.post_mortem()

In [ ]:
# Debugging with custom exception hooks
import sys

# Store the original excepthook
original_excepthook = sys.excepthook

# Define a custom exception handler
def custom_excepthook(exc_type, exc_value, exc_traceback):
    print("\n===== Custom Exception Handler =====")
    print(f"Exception type: {exc_type.__name__}")
    print(f"Exception message: {exc_value}")
    
    # Get the traceback as a string
    tb_lines = traceback.format_tb(exc_traceback)
    print("\nTraceback:")
    for line in tb_lines:
        print(line, end="")
    
    print("\n===== End of Custom Handler =====")

# Set the custom exception hook
sys.excepthook = custom_excepthook

# A function that will raise an exception
def divide_numbers():
    return 10 / 0

try:
    # This will raise an exception and trigger our custom handler
    divide_numbers()
except:
    # We'll catch it here to continue execution in the notebook
    pass

# Restore the original excepthook
sys.excepthook = original_excepthook

In [ ]:
# Debug using context variables
class DebugContext:
    def __init__(self, debug_enabled=False):
        self.debug_enabled = debug_enabled
        self.debug_info = []
    
    def log(self, message):
        if self.debug_enabled:
            self.debug_info.append(message)
    
    def print_debug_info(self):
        if not self.debug_info:
            print("No debug information collected")
            return
        print("\n===== Debug Information =====")
        for i, message in enumerate(self.debug_info):
            print(f"{i+1}. {message}")
    
    def clear(self):
        self.debug_info = []

# Function using debug context
def process_with_debug(numbers, debug_ctx):
    debug_ctx.log(f"Starting processing with {len(numbers)} items")
    
    results = []
    for i, num in enumerate(numbers):
        debug_ctx.log(f"Processing item {i}: {num}")
        
        try:
            if num < 0:
                debug_ctx.log(f"Negative number found: {num}")
                continue
            
            result = num ** 2
            results.append(result)
            debug_ctx.log(f"Processed result: {result}")
        except Exception as e:
            debug_ctx.log(f"Error processing item {i}: {e}")
    
    debug_ctx.log(f"Processing complete. {len(results)} results generated.")
    return results

# Create a debug context and process some data
debug_ctx = DebugContext(debug_enabled=True)
numbers = [5, -3, 2, 0, 10, -7, 8]
results = process_with_debug(numbers, debug_ctx)

print(f"Results: {results}")
debug_ctx.print_debug_info()

## 14. Third-Party Debugging Tools

Several third-party tools can enhance your debugging experience.

In [ ]:
# Using Rich for better traceback formatting
try:
    from rich.traceback import install
    install(show_locals=True)
    
    # A function with an error
    def calculate_with_error(x, y):
        a = x * 2
        b = y - 5
        c = a / b  # This will fail when b is 5
        return c
    
    try:
        result = calculate_with_error(10, 5)
    except Exception as e:
        print(f"Caught exception: {e}")
except ImportError:
    print("Rich package not installed. Install it with 'pip install rich'")

In [ ]:
# Using Loguru for enhanced logging
try:
    from loguru import logger
    
    # Configure logger
    logger.remove()  # Remove default handler
    logger.add(sys.stderr, format="<green>{time}</green> <level>{level}</level> <level>{message}</level>", level="INFO")
    logger.add("loguru.log", rotation="500 KB", level="DEBUG")
    
    # Log various messages
    logger.debug("This is a debug message")
    logger.info("This is an info message")
    logger.warning("This is a warning message")
    logger.error("This is an error message")
    
    # Log exceptions
    try:
        1/0
    except Exception as e:
        logger.exception("An error occurred")
    
    # Structured logging
    logger.info("User {user} logged in from {ip}", user="john_doe", ip="192.168.1.1")
    
except ImportError:
    print("Loguru package not installed. Install it with 'pip install loguru'")

## Summary

This notebook covered comprehensive techniques for logging and debugging in Python:

1. **Setting Up Python Logging**: Configuring the basic logging system
2. **Logging Levels and Basic Configuration**: Understanding and using different severity levels
3. **Configuring Log Handlers and Formatters**: Directing logs to different outputs with custom formats
4. **Logging to Files**: Persisting logs for future analysis
5. **Creating Custom Loggers**: Building specialized loggers for different components
6. **Logging in Applications and Modules**: Best practices for using logging in larger codebases
7. **Exception Handling with Logging**: Capturing and logging errors properly
8. **Using the Python Debugger (pdb)**: Interactive debugging with pdb
9. **Debugging with Breakpoints**: Advanced breakpoint techniques
10. **Assertions for Debugging**: Using assertions to catch logical errors
11. **Using Logging for Debugging**: Leveraging logs for non-interactive debugging
12. **Performance Profiling**: Identifying performance bottlenecks
13. **Advanced Debugging Techniques**: Custom exception hooks and contextual debugging
14. **Third-Party Debugging Tools**: Enhancing debugging with additional libraries

Effective logging and debugging are essential skills for developing reliable Python applications. By mastering these techniques, you can significantly reduce the time spent tracking down bugs and improve the quality of your code.